In [27]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [28]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
736,"I went to see Random Hearts with 3 friends, an...",negative
791,This is a great British film. A cleverly obser...,positive
701,"Based on an actual story, John Boorman shows t...",positive
665,"Unfortunately, one of the best efforts yet mad...",negative
316,"This has to be one of the most beautiful, movi...",positive


In [29]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [30]:
df = normalize_text(df)
df.head()

,review,sentiment
736,went see random heart friend first thought may...,negative
791,great british film cleverly observed script ma...,positive
701,based actual story john boorman show struggle ...,positive
665,unfortunately one best effort yet made area sp...,negative
316,one beautiful moving thought provoking film ar...,positive


In [31]:

df['sentiment'].value_counts()

sentiment
positive    254
negative    246
Name: count, dtype: int64

In [32]:

x = df['sentiment'].isin(['positive','negative'])
df = df[x]


In [33]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
736,went see random heart friend first thought may...,0
791,great british film cleverly observed script ma...,1
701,based actual story john boorman show struggle ...,1
665,unfortunately one best effort yet made area sp...,0
316,one beautiful moving thought provoking film ar...,1


In [34]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [35]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [37]:
import dagshub

mlflow.set_tracking_uri('https://dagshub.com/Suhas-124/capstone-project.mlflow')
dagshub.init(repo_owner='Suhas-124', repo_name='capstone-project', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("Logistic Regression Baseline")

2025-05-01 12:14:14,112 - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/Suhas-124/capstone-project "HTTP/1.1 200 OK"


Initialized MLflow to track repo "Suhas-124/capstone-project"

2025-05-01 12:14:14,128 - INFO - Initialized MLflow to track repo "Suhas-124/capstone-project"


Repository Suhas-124/capstone-project initialized!

2025-05-01 12:14:14,132 - INFO - Repository Suhas-124/capstone-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/9a1808aa4fd848319a4b9788625681a2', creation_time=1746081186016, experiment_id='0', last_update_time=1746081186016, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}>

In [38]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.30)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        notebook_path = "exp1_baseline_model.ipynb"
        logging.info("Executing Jupyter Notebook. This may take a while...")
        os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        mlflow.log_artifact(notebook_path)

        logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)

2025-05-01 12:14:14,562 - INFO - Starting MLflow run...
2025-05-01 12:14:14,902 - INFO - Logging preprocessing parameters...
2025-05-01 12:14:15,962 - INFO - Initializing Logistic Regression model...
2025-05-01 12:14:15,963 - INFO - Fitting the model...
2025-05-01 12:14:16,171 - INFO - Model training complete.
2025-05-01 12:14:16,172 - INFO - Logging model parameters...
2025-05-01 12:14:16,495 - INFO - Making predictions...
2025-05-01 12:14:16,496 - INFO - Calculating evaluation metrics...
2025-05-01 12:14:16,511 - INFO - Logging evaluation metrics...
2025-05-01 12:14:18,001 - INFO - Saving and logging the model...
2025/05/01 12:14:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025-05-01 12:14:24,351 - INFO - Model training and logging completed in 9.45 seconds.
2025-05-01 12:14:24,353 - INFO - Executing Jupyter Notebook. This may take a while...
usage: j

🏃 View run traveling-asp-900 at: https://dagshub.com/Suhas-124/capstone-project.mlflow/#/experiments/0/runs/806dc69cc0314c6b8006e19b61468c8b
🧪 View experiment at: https://dagshub.com/Suhas-124/capstone-project.mlflow/#/experiments/0
